# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 9A – Linguistic Enrichment of the Quran Passage Collection (DD-010)

## Introduction

Notebook 8 preserves and structures the official QuranQA Passage Collection. This
notebook constructs the passage-side linguistic representation used by the root-aware
retrieval stages.

Passage processing is deliberately distinct from query processing. Query roots use
QAC first with CAMeL fallback, whereas passage roots are produced directly through
CAMeL morphological analysis. The passage pipeline is:

**processed QPC → passage normalization/tokenization → CAMeL morphological analysis → validated passage linguistic representation**

The corrected implementation analyzes each normalized vocabulary item once, preserves
the CAMeL decision evidence, applies the approved strict root-validity rule, and maps
passage tokens through that authoritative vocabulary. It does not run a second,
unvalidated analyzer path while constructing passage records.

---

## Objectives

1. Load the processed Quran passage collection produced by Notebook 8.
2. Normalize and tokenize passage text while preserving token provenance.
3. Construct a deterministic normalized passage vocabulary.
4. Apply CAMeL-only morphological analysis and strict root validation.
5. Preserve analysis, ambiguity, raw-root, normalized-root, lemma, stem, and POS evidence.
6. Attach exact Maqāyīs meanings to validated roots.
7. Construct each passage exclusively from the validated vocabulary representation.
8. Export corrected vocabulary, token-provenance, and passage representations.

---

## Design Decision (DD-010)

Passage roots are CAMeL-only. QAC is not used for passage-side fallback or override.
Roots are not accepted by length: complete one- and two-character Arabic roots remain
eligible, while sentinels, missing roots, roots containing `#`, and non-Arabic normalized
roots remain unresolved.

In [1]:
from pathlib import Path
from importlib.metadata import version
import sys
import json
import re
import sqlite3

import numpy as np
import pandas as pd

from IPython.display import display


def find_workspace_root(start_path):
    """Locate the authoritative Writing Workspace without a personal absolute path."""
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        authority_file = (
            candidate /
            "00_Project_Control" /
            "README_Workspace_Authority.txt"
        )
        qpc_file = (
            candidate /
            "02_Methodology_Evidence" /
            "Data and Resources" /
            "QuranQA" /
            "QQA23_TaskA_QPC_v1.1.tsv"
        )

        if authority_file.exists() and qpc_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the authoritative thesis Writing Workspace "
        "from the current execution directory."
    )


NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_workspace_root(NOTEBOOK_DIR)

ROOT_EXTRACTION_DIR = (
    PROJECT_ROOT /
    "02_Methodology_Evidence" /
    "QAC and Root Extraction"
)

MAQAYIS_DIR = (
    PROJECT_ROOT /
    "02_Methodology_Evidence" /
    "Maqāyīs Lookup"
)

for module_dir in [ROOT_EXTRACTION_DIR, MAQAYIS_DIR]:
    module_path = str(module_dir)
    if module_path not in sys.path:
        sys.path.insert(0, module_path)

import arabic_preprocessing as ap
import hybrid_root_extractor as hre
import semantic_lookup as sl

print("=" * 70)
print("PASSAGE-SIDE ENVIRONMENT")
print("=" * 70)
print(f"Python             : {sys.version.split()[0]}")
print(f"CAMeL Tools        : {version('camel-tools')}")
print(f"pandas             : {pd.__version__}")
print(f"NumPy              : {np.__version__}")
print(f"Root helper        : {Path(hre.__file__).resolve()}")
print(f"Preprocess helper  : {Path(ap.__file__).resolve()}")
print(f"Semantic helper    : {Path(sl.__file__).resolve()}")

PASSAGE-SIDE ENVIRONMENT
Python             : 3.12.13
CAMeL Tools        : 1.6.0
pandas             : 3.0.3
NumPy              : 2.5.0
Root helper        : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\hybrid_root_extractor.py
Preprocess helper  : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\arabic_preprocessing.py
Semantic helper    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\semantic_lookup.py


In [2]:
# ============================================================
# Define Current Workspace Paths
# ============================================================

DATA_RESOURCES_DIR = (
    PROJECT_ROOT /
    "02_Methodology_Evidence" /
    "Data and Resources"
)

QURANQA_DIR = DATA_RESOURCES_DIR / "QuranQA"

PASSAGE_PREPARATION_DIR = (
    DATA_RESOURCES_DIR /
    "Corrected_Outputs" /
    "QPC_Passage_Side_Validation"
)

LEXICON_DIR = MAQAYIS_DIR / "Lexicon Database and Metadata"
MAQAYIS_DB_FILE = LEXICON_DIR / "db.sqlite"

PASSAGE_ENRICHMENT_DIR = (
    ROOT_EXTRACTION_DIR /
    "Corrected_Outputs" /
    "QPC_Passage_Side_Validation"
)

PASSAGE_ENRICHMENT_DIR.mkdir(parents=True, exist_ok=True)

PASSAGE_COLLECTION_FILE = (
    PASSAGE_PREPARATION_DIR /
    "QuranQA_Processed_Passage_Collection.csv"
)

OUTPUT_FILE = (
    PASSAGE_ENRICHMENT_DIR /
    "QuranQA_Linguistically_Enriched_Passages.csv"
)

TOKEN_PROVENANCE_FILE = (
    PASSAGE_ENRICHMENT_DIR /
    "QuranQA_Passage_Token_Provenance.csv"
)

PASSAGE_REPRESENTATION_FILE = (
    PASSAGE_ENRICHMENT_DIR /
    "QuranQA_Enriched_Passage_Representation.csv"
)

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)
print(f"Project Root                 : {PROJECT_ROOT}")
print(f"Processed Passage Collection : {PASSAGE_COLLECTION_FILE}")
print(f"Maqayis Database             : {MAQAYIS_DB_FILE}")
print(f"Corrected Output Directory   : {PASSAGE_ENRICHMENT_DIR}")

required_paths = [PASSAGE_COLLECTION_FILE, MAQAYIS_DB_FILE]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required resources: {missing_paths}")

PROJECT PATHS
Project Root                 : ${THESIS_WORKSPACE}
Processed Passage Collection : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Processed_Passage_Collection.csv
Maqayis Database             : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Maqāyīs Lookup\Lexicon Database and Metadata\db.sqlite
Corrected Output Directory   : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation


## Loading the Processed Quran Passage Collection

This section loads the processed Quran passage collection generated in Notebook 8. The dataset serves as the document corpus of the proposed retrieval framework and contains the original Quranic passages together with their structural metadata.

Unlike the query processing pipeline, the linguistic analysis is performed at the vocabulary level rather than independently for every passage. By constructing a unique vocabulary from all passage terms, each lexical item is analyzed only once and subsequently reused across the entire document collection. This approach improves computational efficiency, guarantees consistent linguistic annotation, and follows the lookup-table strategy adopted in the previous notebooks.

In [3]:
# ============================================================
# Load Processed Quran Passage Collection
# ============================================================

passages_df = pd.read_csv(
    PASSAGE_COLLECTION_FILE,
    encoding="utf-8-sig"
)

print("=" * 70)
print("PROCESSED PASSAGE COLLECTION LOADED")
print("=" * 70)

print(f"Total Passages : {len(passages_df):,}")
print(f"Total Columns  : {passages_df.shape[1]}")

print("\nColumns")

print(list(passages_df.columns))

print("\nSample Passages")

display(
    passages_df[
        [
            "Passage_ID",
            "Passage_Text"
        ]
    ].head(5)
)

PROCESSED PASSAGE COLLECTION LOADED
Total Passages : 1,266
Total Columns  : 8

Columns
['Passage_ID', 'Passage_Text', 'Surah', 'Start_Ayah', 'End_Ayah', 'Ayah_Count', 'Character_Count', 'Word_Count']

Sample Passages


,Passage_ID,Passage_Text
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين....
1,1:5-6,إياك نعبد وإياك نستعين. اهدنا الصراط المستقيم.
2,1:7-7,صراط الذين أنعمت عليهم غير المغضوب عليهم ولا ا...
3,2:1-2,الم. ذلك الكتاب لا ريب فيه هدى للمتقين.
4,2:3-5,الذين يؤمنون بالغيب ويقيمون الصلاة ومما رزقناه...


## Preprocessing the Quran Passage Collection

To ensure methodological consistency, the Quran passages are preprocessed using the same linguistic normalization principles applied to the user queries in Notebook 4. The preprocessing stage prepares each passage for lexical analysis by normalizing Arabic orthography, removing diacritics and punctuation, and tokenizing the text into individual lexical items.

Applying a common preprocessing pipeline to both queries and passages ensures that subsequent root extraction and semantic enrichment operate on comparable lexical representations, thereby improving the consistency of the proposed retrieval framework.

In [4]:
# ============================================================
# Preprocess the Quran Passage Collection with Token Provenance
# ============================================================


def original_surface_tokens(text):
    """Tokenize after punctuation removal but before orthographic normalization."""
    return ap.tokenize(ap.remove_punctuation(text))


passages_df = passages_df.copy()

passages_df["Original_Tokens"] = (
    passages_df["Passage_Text"]
    .apply(original_surface_tokens)
)

passages_df["Normalized_Text"] = (
    passages_df["Passage_Text"]
    .apply(ap.normalize_arabic)
    .apply(ap.remove_punctuation)
)

passages_df["Tokens"] = (
    passages_df["Normalized_Text"]
    .apply(ap.tokenize)
)

token_records = []

for _, row in passages_df.iterrows():
    original_tokens = row["Original_Tokens"]
    normalized_tokens = row["Tokens"]

    if len(original_tokens) != len(normalized_tokens):
        raise ValueError(
            f"Token provenance length mismatch for passage {row['Passage_ID']}: "
            f"{len(original_tokens)} original vs {len(normalized_tokens)} normalized"
        )

    for position, (original_token, normalized_token) in enumerate(
        zip(original_tokens, normalized_tokens),
        start=1
    ):
        token_records.append({
            "Passage_ID": row["Passage_ID"],
            "Token_Position": position,
            "Original_Token": original_token,
            "Normalized_Token": normalized_token,
        })

token_provenance_df = pd.DataFrame(token_records)

total_tokens = len(token_provenance_df)

print("=" * 70)
print("PASSAGE COLLECTION PREPROCESSED")
print("=" * 70)
print(f"Total Passages          : {len(passages_df):,}")
print(f"Total Token Occurrences : {total_tokens:,}")
print(f"Token Mapping Mismatches: 0")

display(
    token_provenance_df.head(10)
)

PASSAGE COLLECTION PREPROCESSED
Total Passages          : 1,266
Total Token Occurrences : 77,909
Token Mapping Mismatches: 0


,Passage_ID,Token_Position,Original_Token,Normalized_Token
0,1:1-4,1,بسم,بسم
1,1:1-4,2,الله,الله
2,1:1-4,3,الرحمن,الرحمن
3,1:1-4,4,الرحيم,الرحيم
4,1:1-4,5,الحمد,الحمد
5,1:1-4,6,لله,لله
6,1:1-4,7,رب,رب
7,1:1-4,8,العالمين,العالمين
8,1:1-4,9,الرحمن,الرحمن
9,1:1-4,10,الرحيم,الرحيم


## Constructing the Quran Lexical Vocabulary

Following preprocessing, the normalized lexical tokens extracted from all Quran passages are aggregated to construct a unified vocabulary. Each unique lexical item is recorded together with its frequency across the passage collection.

Building the vocabulary after preprocessing ensures that orthographic variants have already been normalized, producing a consistent lexical inventory for subsequent hybrid root retrieval and semantic enrichment. The vocabulary serves as the lexical index of the document collection and forms the basis for the document-side linguistic representation.

In [5]:
# ============================================================
# Construct the Deterministic Quran Lexical Vocabulary
# ============================================================

from collections import Counter

all_tokens = token_provenance_df["Normalized_Token"].tolist()
token_counter = Counter(all_tokens)

original_form_lookup = {}
for normalized_token, group in token_provenance_df.groupby(
    "Normalized_Token",
    sort=False
):
    original_form_lookup[normalized_token] = list(
        dict.fromkeys(group["Original_Token"].tolist())
    )

vocabulary_df = pd.DataFrame(
    token_counter.items(),
    columns=["Word", "Frequency"]
)

vocabulary_df = (
    vocabulary_df
    .sort_values(
        by=["Frequency", "Word"],
        ascending=[False, True],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

vocabulary_df["Normalized_Token"] = vocabulary_df["Word"]
vocabulary_df["Original_Token_Forms"] = vocabulary_df["Word"].apply(
    lambda token: json.dumps(
        original_form_lookup[token],
        ensure_ascii=False,
        allow_nan=False
    )
)

print("=" * 70)
print("QURAN LEXICAL VOCABULARY CREATED")
print("=" * 70)
print(f"Total Token Occurrences : {len(all_tokens):,}")
print(f"Unique Lexical Terms    : {len(vocabulary_df):,}")

display(vocabulary_df.head(20))

QURAN LEXICAL VOCABULARY CREATED
Total Token Occurrences : 77,909
Unique Lexical Terms    : 14,693


,Word,Frequency,Normalized_Token,Original_Token_Forms
0,من,2769,من,"[""من""]"
1,الله,2156,الله,"[""الله"", ""آلله""]"
2,ان,1607,ان,"[""إن"", ""أن"", ""آن""]"
3,في,1186,في,"[""في""]"
4,ما,1011,ما,"[""ما""]"
5,لا,812,لا,"[""لا""]"
6,الذين,810,الذين,"[""الذين""]"
7,الا,763,الا,"[""إلا"", ""ألا""]"
8,علي,686,علي,"[""على"", ""علي""]"
9,ولا,658,ولا,"[""ولا""]"


## Preparing the Vocabulary for CAMeL Morphological Analysis

The normalized passage vocabulary is analyzed directly with CAMeL Tools. Each unique
normalized item is analyzed once, and all passage occurrences later refer back to that
single vocabulary decision. QAC is not loaded or consulted on the passage side.

The vocabulary preserves original surface variants, normalized CAMeL input, analysis
counts, selected-analysis evidence, tie information, raw and normalized roots, validity,
unresolved reason, lemma, stem, POS, and source.

In [6]:
# ============================================================
# Prepare Vocabulary Provenance Columns
# ============================================================

lexical_vocabulary_df = vocabulary_df.copy()

lexical_vocabulary_df.insert(
    0,
    "Vocabulary_ID",
    range(1, len(lexical_vocabulary_df) + 1)
)

defaults = {
    "CAMeL_Input_Token": None,
    "Analysis_Source": "CAMeL",
    "Analysis_Found": False,
    "Analysis_Count": 0,
    "Analysis_Score": None,
    "Top_Analysis_Count": 0,
    "Top_Roots": "[]",
    "Top_Root_Validations": "[]",
    "Selected_Analysis_Evidence": "null",
    "Root_Ambiguous": False,
    "Resolution_Status": None,
    "Root_Unresolved_Reason": None,
    "CAMeL_Root_Raw": None,
    "CAMeL_Root_Normalized": None,
    "CAMeL_Root_Valid": False,
    "CAMeL_Root_Invalid_Reason": None,
    "Root_AR": None,
    "Hybrid_Root": None,
    "Lemma_AR": None,
    "Stem_AR": None,
    "POS": None,
    "Root_Source": None,
    "Root_Resolved": False,
    "Root_Found": False,
    "Semantic_Meaning": None,
}

for column, default in defaults.items():
    lexical_vocabulary_df[column] = default

print("=" * 70)
print("LEXICAL VOCABULARY PREPARED")
print("=" * 70)
print(f"Vocabulary Entries : {len(lexical_vocabulary_df):,}")
print(f"Columns            : {len(lexical_vocabulary_df.columns):,}")

display(lexical_vocabulary_df.head(10))

LEXICAL VOCABULARY PREPARED
Vocabulary Entries : 14,693
Columns            : 30


,Vocabulary_ID,Word,Frequency,Normalized_Token,Original_Token_Forms,CAMeL_Input_Token,Analysis_Source,Analysis_Found,Analysis_Count,Analysis_Score,...,CAMeL_Root_Invalid_Reason,Root_AR,Hybrid_Root,Lemma_AR,Stem_AR,POS,Root_Source,Root_Resolved,Root_Found,Semantic_Meaning
0,1,من,2769,من,"[""من""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
1,2,الله,2156,الله,"[""الله"", ""آلله""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
2,3,ان,1607,ان,"[""إن"", ""أن"", ""آن""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
3,4,في,1186,في,"[""في""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
4,5,ما,1011,ما,"[""ما""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
5,6,لا,812,لا,"[""لا""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
6,7,الذين,810,الذين,"[""الذين""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
7,8,الا,763,الا,"[""إلا"", ""ألا""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
8,9,علي,686,علي,"[""على"", ""علي""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None
9,10,ولا,658,ولا,"[""ولا""]",None,CAMeL,False,0,None,...,None,None,None,None,None,None,None,False,False,None


## CAMeL-Only Root Extraction and Deterministic Selection

Each normalized passage-vocabulary item is analyzed directly with CAMeL Tools. The
Point-2 scoring criteria are retained. Equal top analyses are handled deterministically
when they agree on one valid root. Conflicting equally preferred valid roots are marked
ambiguous and remain unresolved rather than being selected by analyzer order.

A root is valid only when its raw value is present, is not an analyzer sentinel, does
not contain `#`, and becomes Arabic letters only after permitted dot-separator and
diacritic removal. Root length is not a rejection criterion.

In [7]:
# ============================================================
# Analyze the Quran Lexical Vocabulary with CAMeL Tools
# ============================================================


def strict_json(value):
    """Serialize provenance with standard JSON null values, never NaN."""
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        allow_nan=False,
        default=str
    )


selected_evidence_fields = [
    "root",
    "lex",
    "stem",
    "pos",
    "bw",
    "gloss",
    "source",
    "pattern",
]

for idx, row in lexical_vocabulary_df.iterrows():
    token = row["Word"]
    camel_input = hre.normalize_for_camel(token)
    analyses = hre.CAMEL_ANALYZER.analyze(camel_input)
    selection = hre.select_best_analysis_with_provenance(token, analyses)

    analysis = selection["Selected_Analysis"]
    root_validation = selection["Selected_Root_Validation"]

    if analysis is None:
        selected_evidence = None
    else:
        selected_evidence = {
            field: analysis.get(field)
            for field in selected_evidence_fields
            if analysis.get(field) is not None
        }

    if root_validation is None:
        raw_root = None
        normalized_root = None
        root_valid = False
        invalid_reason = None
    else:
        raw_root = root_validation["Raw"]
        normalized_root = root_validation["Normalized"]
        root_valid = bool(root_validation["Valid"])
        invalid_reason = root_validation["Invalid_Reason"]

    root_resolved = bool(
        selection["Root_Resolved"]
        and root_valid
        and normalized_root
    )

    if root_resolved:
        unresolved_reason = None
    elif selection["Status"] == "NO_ANALYSIS":
        unresolved_reason = "no_analysis"
    elif selection["Status"] == "AMBIGUOUS_TOP_ROOT":
        unresolved_reason = "ambiguous_top_root"
    else:
        unresolved_reason = invalid_reason or "unresolved_root"

    lexical_vocabulary_df.at[idx, "CAMeL_Input_Token"] = camel_input
    lexical_vocabulary_df.at[idx, "Analysis_Found"] = bool(selection["Analysis_Found"])
    lexical_vocabulary_df.at[idx, "Analysis_Count"] = int(selection["Analysis_Count"])
    lexical_vocabulary_df.at[idx, "Analysis_Score"] = selection["Top_Score"]
    lexical_vocabulary_df.at[idx, "Top_Analysis_Count"] = int(selection["Top_Analysis_Count"])
    lexical_vocabulary_df.at[idx, "Top_Roots"] = strict_json(selection["Top_Roots"])
    lexical_vocabulary_df.at[idx, "Top_Root_Validations"] = strict_json(selection["Top_Root_Validations"])
    lexical_vocabulary_df.at[idx, "Selected_Analysis_Evidence"] = strict_json(selected_evidence)
    lexical_vocabulary_df.at[idx, "Root_Ambiguous"] = bool(selection["Root_Ambiguous"])
    lexical_vocabulary_df.at[idx, "Resolution_Status"] = selection["Status"]
    lexical_vocabulary_df.at[idx, "Root_Unresolved_Reason"] = unresolved_reason
    lexical_vocabulary_df.at[idx, "CAMeL_Root_Raw"] = raw_root
    lexical_vocabulary_df.at[idx, "CAMeL_Root_Normalized"] = normalized_root
    lexical_vocabulary_df.at[idx, "CAMeL_Root_Valid"] = root_valid
    lexical_vocabulary_df.at[idx, "CAMeL_Root_Invalid_Reason"] = invalid_reason
    lexical_vocabulary_df.at[idx, "Root_AR"] = normalized_root if root_resolved else None
    lexical_vocabulary_df.at[idx, "Hybrid_Root"] = normalized_root if root_resolved else None
    lexical_vocabulary_df.at[idx, "Lemma_AR"] = analysis.get("lex") if analysis else None
    lexical_vocabulary_df.at[idx, "Stem_AR"] = analysis.get("stem") if analysis else None
    lexical_vocabulary_df.at[idx, "POS"] = analysis.get("pos") if analysis else None
    lexical_vocabulary_df.at[idx, "Root_Source"] = "CAMeL" if root_resolved else None
    lexical_vocabulary_df.at[idx, "Root_Resolved"] = root_resolved
    lexical_vocabulary_df.at[idx, "Root_Found"] = root_resolved

vocabulary_id_lookup = dict(zip(
    lexical_vocabulary_df["Word"],
    lexical_vocabulary_df["Vocabulary_ID"]
))

token_provenance_df["Vocabulary_ID"] = (
    token_provenance_df["Normalized_Token"]
    .map(vocabulary_id_lookup)
)

if token_provenance_df["Vocabulary_ID"].isna().any():
    raise ValueError("At least one passage token has no vocabulary record")

token_provenance_df["Vocabulary_ID"] = (
    token_provenance_df["Vocabulary_ID"].astype(int)
)

print("=" * 70)
print("CAMEL VOCABULARY ANALYSIS COMPLETED")
print("=" * 70)
print(f"Vocabulary Entries  : {len(lexical_vocabulary_df):,}")
print(f"Analyses Examined   : {lexical_vocabulary_df['Analysis_Count'].sum():,}")
print(f"Valid Roots         : {lexical_vocabulary_df['Root_Resolved'].sum():,}")
print(f"Unresolved Terms    : {(~lexical_vocabulary_df['Root_Resolved']).sum():,}")

CAMEL VOCABULARY ANALYSIS COMPLETED
Vocabulary Entries  : 14,693
Analyses Examined   : 125,452
Valid Roots         : 8,684
Unresolved Terms    : 6,009


## Validate Passage-Side CAMeL Roots

The CAMeL decision is validated before any passage representation is built. Raw analyzer
evidence remains preserved. Sentinel values and incomplete `#` roots are unresolved;
dots and Arabic diacritics are removed only in the normalized value. Complete Arabic
one- and two-character roots are retained because length alone is not a validity rule.

In [8]:
# ============================================================
# Validate Strict CAMeL Root Decisions and Report Diagnostics
# ============================================================

resolved_mask = lexical_vocabulary_df["Root_Resolved"].astype(bool)
root_nonblank = (
    lexical_vocabulary_df["Root_AR"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)
source_present = (
    lexical_vocabulary_df["Root_Source"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

if (resolved_mask & ~root_nonblank).any():
    raise AssertionError("Resolved vocabulary record has a blank root")
if (resolved_mask & ~source_present).any():
    raise AssertionError("Resolved vocabulary record has no root source")
if (resolved_mask != lexical_vocabulary_df["Root_Found"].astype(bool)).any():
    raise AssertionError("Root_Found compatibility field disagrees with Root_Resolved")
if (resolved_mask & ~lexical_vocabulary_df["CAMeL_Root_Valid"].astype(bool)).any():
    raise AssertionError("Resolved vocabulary record failed strict CAMeL validation")

invalid_reason_counts = (
    lexical_vocabulary_df["CAMeL_Root_Invalid_Reason"]
    .dropna()
    .value_counts()
    .to_dict()
)

status_counts = (
    lexical_vocabulary_df["Resolution_Status"]
    .value_counts(dropna=False)
    .to_dict()
)

raw_roots = lexical_vocabulary_df["CAMeL_Root_Raw"].fillna("").astype(str)
normalized_roots = lexical_vocabulary_df["Root_AR"].fillna("").astype(str)

diacritic_pattern = re.compile(
    r"[ؐ-ًؚ-ٰٟۖ-ۭ]"
)

diacritic_retained_mask = (
    resolved_mask &
    raw_roots.str.contains(diacritic_pattern, regex=True)
)

one_character_mask = resolved_mask & normalized_roots.str.len().eq(1)
two_character_mask = resolved_mask & normalized_roots.str.len().eq(2)

print("=" * 70)
print("STRICT PASSAGE-SIDE ROOT VALIDATION")
print("=" * 70)
print(f"Valid-root vocabulary terms       : {resolved_mask.sum():,}")
print(f"Unresolved vocabulary terms       : {(~resolved_mask).sum():,}")
print(f"Sentinel-root rejections          : {invalid_reason_counts.get('sentinel_root', 0):,}")
print(f"Incomplete-# rejections           : {invalid_reason_counts.get('incomplete_placeholder_root', 0):,}")
print(f"Non-Arabic-root rejections        : {invalid_reason_counts.get('non_arabic_root', 0):,}")
print(f"Missing-root rejections           : {invalid_reason_counts.get('missing_root', 0):,}")
print(f"Diacritic-bearing roots retained  : {diacritic_retained_mask.sum():,}")
print(f"Valid one-character roots         : {one_character_mask.sum():,}")
print(f"Valid two-character roots         : {two_character_mask.sum():,}")

print("\nResolution Status")
display(pd.DataFrame(
    sorted(status_counts.items()),
    columns=["Resolution_Status", "Count"]
))

print("\nOne-character valid-root examples")
display(lexical_vocabulary_df.loc[
    one_character_mask,
    ["Word", "CAMeL_Root_Raw", "Root_AR", "Lemma_AR", "POS"]
].head(10))

print("\nTwo-character valid-root examples")
display(lexical_vocabulary_df.loc[
    two_character_mask,
    ["Word", "CAMeL_Root_Raw", "Root_AR", "Lemma_AR", "POS"]
].head(10))

STRICT PASSAGE-SIDE ROOT VALIDATION
Valid-root vocabulary terms       : 8,684
Unresolved vocabulary terms       : 6,009
Sentinel-root rejections          : 144
Incomplete-# rejections           : 4,631
Non-Arabic-root rejections        : 0
Missing-root rejections           : 1
Diacritic-bearing roots retained  : 38
Valid one-character roots         : 78
Valid two-character roots         : 113

Resolution Status


,Resolution_Status,Count
0,AMBIGUOUS_TOP_ROOT,255
1,ANALYSIS_INVALID_ROOT,4775
2,ANALYSIS_NO_ROOT,1
3,NO_ANALYSIS,978
4,RESOLVED,8684



One-character valid-root examples


,Word,CAMeL_Root_Raw,Root_AR,Lemma_AR,POS
19,به,ب,ب,بِ,prep
20,وان,ن,ن,وَإِن,conj_sub
26,له,ل,ل,لِ,prep
28,هو,ه,ه,هُوَ,pron
34,والله,ل,ل,لِ,prep
43,هذا,ه,ه,هٰذا,pron_dem
50,وهو,ه,ه,هُوَ,pron
56,اني,نّ,ن,أَنَّى,conj
60,ايها,ّه,ه,أَيّ,pron_interrog
65,بالله,ل,ل,لِ,prep



Two-character valid-root examples


,Word,CAMeL_Root_Raw,Root_AR,Lemma_AR,POS
0,من,م.ن,من,مَن,pron_interrog
15,ومن,م.ن,من,مَن,pron_interrog
21,بما,بم,بم,ما,part_interrog
36,عن,ع.ن,عن,عَن,prep
59,منهم,م.ن,من,مِن,prep
84,مما,ممّ,مم,مِمّا,conj
91,منكم,م.ن,من,مِن,prep
99,لمن,م.ن,من,مَن,pron_interrog
100,دون,دن,دن,دُونَ,prep
101,فمن,م.ن,من,مَن,pron_interrog


# Semantic Enrichment Using Maqāyīs al-Lughah

The validated Arabic roots extracted from the Qur'anic lexical
vocabulary are semantically enriched using the classical Arabic
lexicon *Maqāyīs al-Lughah* by Ibn Fāris.

A semantic lookup dictionary is constructed from the lexicon and
used to retrieve the primary semantic meaning associated with each
validated Arabic root.

The resulting semantic descriptions are integrated into the
linguistically enriched vocabulary, providing a richer lexical
representation for the subsequent retrieval stage.

In [9]:
# ============================================================
# Load Maqāyīs al-Lughah Semantic Lookup
# ============================================================

connection = sqlite3.connect(MAQAYIS_DB_FILE)

maqayis_df = pd.read_sql_query(
    """
    SELECT word, meanings
    FROM maqayeesul_luga
    """,
    connection
)

connection.close()

maqayis_lookup, duplicate_entries = sl.build_maqayis_lookup(
    maqayis_df
)

print("=" * 70)
print("MAQAYIS SEMANTIC LOOKUP CREATED")
print("=" * 70)

print(f"Dictionary Entries : {len(maqayis_lookup):,}")
print(f"Duplicate Entries  : {duplicate_entries:,}")

print("\nSample Dictionary Entries")

sample_df = (
    pd.DataFrame(
        list(maqayis_lookup.items()),
        columns=["Root_AR", "Semantic_Meaning"]
    )
)

display(sample_df.head(10))

MAQAYIS SEMANTIC LOOKUP CREATED
Dictionary Entries : 5,274
Duplicate Entries  : 0

Sample Dictionary Entries


,Root_AR,Semantic_Meaning
0,أب,(أَبَّ) اعْلَمْ أَنَّ لِلْهَمْزَةِ وَالْبَاءِ ...
1,أبت,(أَبَتَ) الْهَمْزَةُ وَالْبَاءُ وَالتَّاءُ أَص...
2,أبث,(أَبَثَ) وَهَذَا الْبَابُ مُهْمَلٌ عِنْدَ الْخ...
3,أبد,(أَبَدَ) الْهَمْزَةُ وَالْبَاءُ وَالدَّالُ يَد...
4,أبر,(أَبَرَ) الْهَمْزَةُ وَالْبَاءُ وَالرَّاءُ يَد...
5,أبز,(أَبَزَ) الْهَمْزَةُ وَالْبَاءُ وَالزَّاءُ يَد...
6,أبس,(أَبَسَ) الْهَمْزَةُ وَالْبَاءُ وَالسِّينُ تَد...
7,أبش,(أَبَشَ) الْهَمْزَةُ وَالْبَاءُ وَالشِّينُ لَي...
8,أبض,(أَبَضَ) الْهَمْزَةُ وَالْبَاءُ وَالضَّادُ تَد...
9,أبط,(أَبَطَ) الْهَمْزَةُ وَالْبَاءُ وَالطَّاءُ أَص...


# Semantic Enrichment of the Qur'anic Lexical Vocabulary

Each validated Arabic root is matched against the semantic lookup
dictionary constructed from *Maqāyīs al-Lughah*.

When a matching root is found, its primary semantic meaning is added
to the Qur'anic lexical vocabulary. This enriches the vocabulary with
classical lexical semantics while preserving the morphological
information obtained during the hybrid root extraction stage.

The resulting linguistically enriched vocabulary combines
morphological and semantic knowledge for use in the retrieval model.

In [10]:
# ============================================================
# Apply Semantic Enrichment
# ============================================================

semantic_matches = 0

for idx, row in lexical_vocabulary_df.iterrows():

    root = row["Hybrid_Root"]

    if pd.isna(root):
        continue

    meaning = maqayis_lookup.get(root)

    if meaning is not None:

        lexical_vocabulary_df.at[idx, "Semantic_Meaning"] = meaning
        semantic_matches += 1

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

coverage = semantic_matches / len(lexical_vocabulary_df) * 100

print("=" * 70)
print("SEMANTIC ENRICHMENT COMPLETED")
print("=" * 70)

print(f"Vocabulary Entries : {len(lexical_vocabulary_df):,}")
print(f"Semantic Matches   : {semantic_matches:,}")
print(f"Coverage           : {coverage:.2f}%")

print("\nSample Enriched Vocabulary")

display(
    lexical_vocabulary_df[
        [
            "Word",
            "Hybrid_Root",
            "Semantic_Meaning"
        ]
    ]
    .dropna(subset=["Semantic_Meaning"])
    .head(20)
)

SEMANTIC ENRICHMENT COMPLETED
Vocabulary Entries : 14,693
Semantic Matches   : 6,656
Coverage           : 45.30%

Sample Enriched Vocabulary


,Word,Hybrid_Root,Semantic_Meaning
0,من,من,(مَنَّ) الْمِيمُ وَالنُّونُ أَصْلَانِ. أَحَدُه...
13,لهم,لهم,(لَهَمَ) اللَّامُ وَالْهَاءُ وَالْمِيمُ أَصْلٌ...
15,ومن,من,(مَنَّ) الْمِيمُ وَالنُّونُ أَصْلَانِ. أَحَدُه...
17,لكم,لكم,(لَكَمَ) اللَّامُ وَالْكَافُ وَالْمِيمُ كَلِمَ...
36,عن,عن,(عَنَّ) الْعَيْنُ وَالنُّونُ أَصْلَانِ، أَحَدُ...
39,ربك,ربك,(رَبَكَ) الرَّاءُ وَالْبَاءُ وَالْكَافُ كَلِمَ...
44,كفروا,كفر,(كَفَرَ) الْكَافُ وَالْفَاءُ وَالرَّاءُ أَصْلٌ...
55,الكتاب,كتب,(كَتَبَ) الْكَافُ وَالتَّاءُ وَالْبَاءُ أَصْلٌ...
59,منهم,من,(مَنَّ) الْمِيمُ وَالنُّونُ أَصْلَانِ. أَحَدُه...
61,عذاب,عذب,(عَذَبَ) الْعَيْنُ وَالذَّالُ وَالْبَاءُ أَصْل...


In [11]:
# ============================================================
# Export Corrected Vocabulary and Token Provenance
# ============================================================

export_df = lexical_vocabulary_df.copy()

export_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

token_provenance_df.to_csv(
    TOKEN_PROVENANCE_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("CORRECTED VOCABULARY AND TOKEN PROVENANCE EXPORTED")
print("=" * 70)
print(f"Vocabulary Entries : {len(export_df):,}")
print(f"Token Records      : {len(token_provenance_df):,}")
print(f"Vocabulary File    : {OUTPUT_FILE}")
print(f"Token Trace File   : {TOKEN_PROVENANCE_FILE}")

CORRECTED VOCABULARY AND TOKEN PROVENANCE EXPORTED
Vocabulary Entries : 14,693
Token Records      : 77,909
Vocabulary File    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Linguistically_Enriched_Passages.csv
Token Trace File   : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Passage_Token_Provenance.csv


## Discussion: Corrected Morphological and Semantic Coverage

The corrected counts are produced directly by the executed diagnostic cells and are
preserved in the validation report. Root coverage counts only terms with a strict valid
CAMeL root. Analyses with sentinels, `#` placeholders, missing roots, non-Arabic roots,
or conflicting equally preferred roots remain unresolved.

Maqāyīs enrichment continues to use exact matching against validated normalized roots.
Unmatched roots remain available as valid morphological evidence without a fabricated
semantic description. No fuzzy lookup, manual root repair, QAC fallback, or additional
disambiguation model is introduced.

## Construct the Authoritative Passage Representation

Each normalized passage-token occurrence is linked to a single validated vocabulary
record through `Vocabulary_ID`. Passage roots, lemmas, POS values, validity decisions,
and semantic meanings therefore originate from the preserved vocabulary analysis.

The passage representation retains original and normalized token sequences, parallel
vocabulary identifiers, unique validated roots, and unique semantic meanings. CAMeL is
not called again in this stage, eliminating the former second unvalidated analysis path.

In [12]:
# ============================================================
# Construct Passages Exclusively from the Validated Vocabulary
# ============================================================

vocabulary_lookup = (
    lexical_vocabulary_df
    .set_index("Vocabulary_ID")
    .to_dict("index")
)

token_groups = {
    passage_id: group.sort_values("Token_Position", kind="mergesort")
    for passage_id, group in token_provenance_df.groupby("Passage_ID", sort=False)
}

passage_records = []

for _, row in passages_df.iterrows():
    passage_id = row["Passage_ID"]
    group = token_groups.get(passage_id)

    if group is None:
        raise ValueError(f"No token provenance found for passage {passage_id}")

    original_tokens = group["Original_Token"].tolist()
    normalized_tokens = group["Normalized_Token"].tolist()
    vocabulary_ids = group["Vocabulary_ID"].astype(int).tolist()

    roots = []
    semantic_meanings = []
    resolved_token_count = 0

    for vocabulary_id in vocabulary_ids:
        vocabulary_record = vocabulary_lookup[vocabulary_id]

        if not bool(vocabulary_record["Root_Resolved"]):
            continue

        root = vocabulary_record["Root_AR"]
        if root is None or str(root).strip() == "":
            raise AssertionError(
                f"Resolved vocabulary record {vocabulary_id} has no root"
            )

        resolved_token_count += 1

        if root not in roots:
            roots.append(root)

        meaning = vocabulary_record["Semantic_Meaning"]
        if meaning is not None and not pd.isna(meaning):
            if meaning not in semantic_meanings:
                semantic_meanings.append(meaning)

    passage_records.append({
        "Passage_ID": passage_id,
        "Surah": row["Surah"],
        "Start_Ayah": row["Start_Ayah"],
        "End_Ayah": row["End_Ayah"],
        "Passage_Text": row["Passage_Text"],
        "Original_Tokens": strict_json(original_tokens),
        "Tokens": strict_json(normalized_tokens),
        "Token_Vocabulary_IDs": strict_json(vocabulary_ids),
        "Roots": strict_json(roots),
        "Semantic_Meanings": strict_json(semantic_meanings),
        "Token_Count": len(normalized_tokens),
        "Resolved_Token_Count": resolved_token_count,
        "Unresolved_Token_Count": len(normalized_tokens) - resolved_token_count,
        "Root_Count": len(roots),
        "Semantic_Count": len(semantic_meanings),
    })

passage_representation_df = pd.DataFrame(passage_records)

print("=" * 70)
print("AUTHORITATIVE PASSAGE REPRESENTATION CREATED")
print("=" * 70)
print(f"Total Passages : {len(passage_representation_df):,}")
print(f"Passages with Valid Roots : {(passage_representation_df['Root_Count'] > 0).sum():,}")
print(f"Passages with No Valid Roots: {(passage_representation_df['Root_Count'] == 0).sum():,}")

display(passage_representation_df.head())

AUTHORITATIVE PASSAGE REPRESENTATION CREATED
Total Passages : 1,266
Passages with Valid Roots : 1,266
Passages with No Valid Roots: 0


,Passage_ID,Surah,Start_Ayah,End_Ayah,Passage_Text,Original_Tokens,Tokens,Token_Vocabulary_IDs,Roots,Semantic_Meanings,Token_Count,Resolved_Token_Count,Unresolved_Token_Count,Root_Count,Semantic_Count
0,1:1-4,1,1,4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين....,"[""بسم"", ""الله"", ""الرحمن"", ""الرحيم"", ""الحمد"", ""...","[""بسم"", ""الله"", ""الرحمن"", ""الرحيم"", ""الحمد"", ""...","[2954, 2, 220, 288, 372, 82, 72, 159, 220, 288...","[""بسم"", ""رحم"", ""حمد"", ""لله"", ""ربب"", ""علم"", ""ملك""]","[""(بَسَمَ) الْبَاءُ وَالسِّينُ وَالْمِيمُ أَصْ...",13,10,3,7,5
1,1:5-6,1,5,6,إياك نعبد وإياك نستعين. اهدنا الصراط المستقيم.,"[""إياك"", ""نعبد"", ""وإياك"", ""نستعين"", ""اهدنا"", ""...","[""اياك"", ""نعبد"", ""واياك"", ""نستعين"", ""اهدنا"", ""...","[7589, 1637, 12817, 12081, 7538, 1473, 1771]","[""ك"", ""عبد"", ""صرط""]","[""(عَبَدَ) الْعَيْنُ وَالْبَاءُ وَالدَّالُ أَص...",7,4,3,3,2
2,1:7-7,1,7,7,صراط الذين أنعمت عليهم غير المغضوب عليهم ولا ا...,"[""صراط"", ""الذين"", ""أنعمت"", ""عليهم"", ""غير"", ""ال...","[""صراط"", ""الذين"", ""انعمت"", ""عليهم"", ""غير"", ""ال...","[314, 7, 1276, 42, 140, 7314, 42, 10, 1475]","[""صرط"", ""نعم"", ""غضب"", ""ضلل""]","[""(صَرَطَ) الصَّادُ وَالرَّاءُ وَالطَّاءُ وَهُ...",9,4,5,4,3
3,2:1-2,2,1,2,الم. ذلك الكتاب لا ريب فيه هدى للمتقين.,"[""الم"", ""ذلك"", ""الكتاب"", ""لا"", ""ريب"", ""فيه"", ""...","[""الم"", ""ذلك"", ""الكتاب"", ""لا"", ""ريب"", ""فيه"", ""...","[113, 26, 56, 6, 556, 76, 252, 530]","[""لمم"", ""ذلل"", ""كتب"", ""هدد""]","[""(كَتَبَ) الْكَافُ وَالتَّاءُ وَالْبَاءُ أَصْ...",8,4,4,4,1
4,2:3-5,2,3,5,الذين يؤمنون بالغيب ويقيمون الصلاة ومما رزقناه...,"[""الذين"", ""يؤمنون"", ""بالغيب"", ""ويقيمون"", ""الصل...","[""الذين"", ""يؤمنون"", ""بالغيب"", ""ويقيمون"", ""الصل...","[7, 109, 762, 5908, 171, 881, 937, 492, 55, 10...","[""رزق"", ""نفق"", ""بم"", ""نزل"", ""من"", ""قبل"", ""همم""...","[""(رَزَقَ) الرَّاءُ وَالزَّاءُ وَالْقَافُ أُصَ...",28,13,15,10,6


In [13]:
# ============================================================
# Export Corrected Passage Representation
# ============================================================

passage_representation_df.to_csv(
    PASSAGE_REPRESENTATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("CORRECTED PASSAGE REPRESENTATION EXPORTED")
print("=" * 70)
print(f"Total Passages : {len(passage_representation_df):,}")
print(f"Output File    : {PASSAGE_REPRESENTATION_FILE}")

CORRECTED PASSAGE REPRESENTATION EXPORTED
Total Passages : 1,266
Output File    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Enriched_Passage_Representation.csv


In [14]:
# ============================================================
# Final Vocabulary-to-Passage Consistency Validation
# ============================================================

inconsistent_passages = []
passage_id_order = passages_df["Passage_ID"].astype(str).tolist()
representation_id_order = (
    passage_representation_df["Passage_ID"].astype(str).tolist()
)

if passage_id_order != representation_id_order:
    raise AssertionError("Passage ID order changed during Notebook 9A")

for _, passage in passage_representation_df.iterrows():
    vocabulary_ids = json.loads(passage["Token_Vocabulary_IDs"])
    actual_roots = json.loads(passage["Roots"])

    expected_roots = []
    for vocabulary_id in vocabulary_ids:
        vocabulary_record = vocabulary_lookup[int(vocabulary_id)]
        if not bool(vocabulary_record["Root_Resolved"]):
            continue
        root = vocabulary_record["Root_AR"]
        if root not in expected_roots:
            expected_roots.append(root)

    if actual_roots != expected_roots:
        inconsistent_passages.append(passage["Passage_ID"])

if inconsistent_passages:
    raise AssertionError(
        f"Vocabulary-to-passage inconsistencies: {inconsistent_passages[:10]}"
    )

print("=" * 70)
print("FINAL PASSAGE-SIDE VALIDATION")
print("=" * 70)
print(f"Passages                               : {len(passage_representation_df):,}")
print(f"Unique Passage IDs                    : {passage_representation_df['Passage_ID'].nunique():,}")
print(f"Vocabulary-to-passage inconsistencies : {len(inconsistent_passages):,}")
print(f"Duplicate Passage IDs                 : {passage_representation_df['Passage_ID'].duplicated().sum():,}")
print("No second CAMeL analysis path was used during passage construction.")

FINAL PASSAGE-SIDE VALIDATION
Passages                               : 1,266
Unique Passage IDs                    : 1,266
Vocabulary-to-passage inconsistencies : 0
Duplicate Passage IDs                 : 0
No second CAMeL analysis path was used during passage construction.
